# Feature Engine - Topic 08 - Create your own transformer.

 <img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%202%20-%20Unit%20Objective.png">

## Topic Objectives

* Create your own transformer that can be arranged into a pipeline


---

<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%204%20-%20Import%20Package%20for%20Learning.png"> 

## Import Packages for Learning

And load our typical packages.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
from sklearn.pipeline import Pipeline

---

<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%2010-%20Lesson%20Content.png">

##  Create your own transformer

<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%206%20-%20Warning.png"> 

What if, for your existing project, you couldn't find a built-in transformer that satisfies your project needs?
* You can create a transformer. Your custom transformer will be a Python Class, which is a topic you are already familiar with!

<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png"> 

Before defining your custom transformer, all transformers in scikit-learn (and scikit-learn compatible libraries, like feature-engine) are implemented as Python classes, each with its own attributes and methods. 
* Our custom transformer (or Class) must be implemented as a class with the same methods, like fit(), transform(), fit_transform() etc. We will inherit these methods using two scikit-learn base classes: TransformerMixin and BaseEstimator. 


<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png"> 

For that, we will need two base transformers from Scikit-learn. 
* `BaseEstimator`: According to the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.base.BaseEstimator.html), it is a "base class for all estimators in scikit-learn". We will not focus on the technical aspects, only the frame, as it contains the core of what a transformer should have.
* `TransformerMixin`: According to the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.base.TransformerMixin.html), it is a Mixin class for all transformers in scikit-learn.

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin

In feature-engine (and scikit-learn), we have a transformer that replaces the missing value with the mean. But let's imagine it didn't, and we want to create `MyCustomTransformerForMeanImputation()`
* Let's follow along with the code's comment to understand the steps

In [3]:
import pandas as pd # to use .mean()

# We will define three methods for the class: _init_, fit and transform
# The fit_transform() will be inherited since we are using BaseEstimator and TransformerMixin

# Define your transformer name, and as an argument inherit the base classes
class MyCustomTransformerForMeanImputation(BaseEstimator, TransformerMixin):

  #### Here, you define the variables you need to parse when you initialize the class
  def __init__(self, variables):
    # We make sure the variables will be a list, even if only one element
    if not isinstance(variables, list): 
      self.variables = [variables]
    else: self.variables = variables

  #### Here is where the learning happens. We perform the operation we are interested in
  #### In this case, calculate the mean
  def fit(self, X, y=None):
   
    # We want to keep the mean value in a dictionary
    self.imputer_dict_ = {}
      
    # loop over each variable, calculate the mean and save it in the dictionary.  
    for feature in self.variables:
        self.imputer_dict_[feature] = X[feature].mean()
    
    return self

  #### Here, you transform the variables based on what you learned in the .fit()
  #### You can transform into the train set, test set or real-time data
  def transform(self, X):
    # loop over the variables and .fillna() in a given feature based on the 
    # mean of a given feature
    for feature in self.variables:
      X[feature].fillna(self.imputer_dict_[feature], inplace=True)
      
    return X

You may create a custom transformer where you don't need to code the ``.fit().`` For example, imagine you want to apply the upper case method to all the variables. You don't need to learn that; you need to execute it.
* Let's create this transformer and call `ConvertUpperCase()`

In [4]:
# The comments relate to the new concepts for this exercise

class ConvertUpperCase(BaseEstimator, TransformerMixin):
  def __init__(self, variables):
    if not isinstance(variables, list): 
      self.variables = [variables]
    else: self.variables = variables

  # We don't need to learn anything here; we just return self
  # We need to do that anyway to be compatible with scikit-learn format
  def fit(self, X, y=None):
      return self

  # Here, we convert the variables using a method called .upper()
  # We loop over all the variables, check if it is an object, and then use a lambda function...
  # ...to apply .upper() to all rows
  def transform(self, X):
    for feature in self.variables:
      if X[feature].dtype == 'object':
        X[feature] = X[feature].apply(lambda x: x.upper())
      else:
        print(f"Warning: {feature} data type should be object to use ConvertUpperCase()")

    return X

We will use the 'Online_Retail' dataset, which contains information on transactions made by customers through an online retail platform. The dataset includes data on the products that were purchased, the quantity of each product, the date and time of each transaction, the price of each product, the unique identifier for each customer who made a purchase, and the country where each customer is located. 
* We check for missing data

In [5]:
df = pd.read_csv('Online_Retail.csv')
df = df.astype({'CustomerID':'object'})
df.isnull().sum()

InvoiceNo         0
StockCode         0
Description      85
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID     2583
Country           0
dtype: int64

And inspect the DataFrame.

In [6]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,547561,23127,DOLLCRAFT GIRL NICOLE,4,3/24/11 8:44,4.95,14129.0,United Kingdom
1,547561,23128,DOLLCRAFT BOY JEAN-PAUL,4,3/24/11 8:44,4.95,14129.0,United Kingdom
2,547561,23126,DOLLCRAFT GIRL AMELIE,4,3/24/11 8:44,4.95,14129.0,United Kingdom
3,547561,23077,DOUGHNUT LIP GLOSS,20,3/24/11 8:44,1.25,14129.0,United Kingdom
4,547561,84375,SET OF 20 KIDS COOKIE CUTTERS,12,3/24/11 8:44,2.10,14129.0,United Kingdom


We are interested in:
* Cleaning the missing data with `MyCustomTransformerForMeanImputation()` on the numerical variables and `CategoricalImputer()` for categorical variables
* Next, we want to make all words from the 'Country' column upper case. We will use our own transformer: ConvertUpperCase()


We set the pipeline using these rules in three steps. Then we run `.fit_transform()`
* Once we inspect the data with .head(), we notice the `'Country'` variable has all letters in upper case!

In [7]:
from feature_engine.imputation import CategoricalImputer

pipeline = Pipeline([
      ( 'custom_transf', MyCustomTransformerForMeanImputation(variables=['Quantity',
                                                                         'UnitPrice'] )),
                     
      ( 'categorical_imputer', CategoricalImputer(imputation_method='missing',
                                                  fill_value='Missing',
                                                  variables=['InvoiceNo', 'StockCode',
                                                             'Description', 'InvoiceDate',
                                                             'CustomerID', 'Country']) ),
      
      ('upper_case' , ConvertUpperCase(variables=['Country'])),
])

df_transformed = pipeline.fit_transform(df)
df_transformed.head()

/tmp/ipykernel_756602/3400558282.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X[feature].fillna(self.imputer_dict_[feature], inplace=True)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,547561,23127,DOLLCRAFT GIRL NICOLE,4,3/24/11 8:44,4.95,14129.0,UNITED KINGDOM
1,547561,23128,DOLLCRAFT BOY JEAN-PAUL,4,3/24/11 8:44,4.95,14129.0,UNITED KINGDOM
2,547561,23126,DOLLCRAFT GIRL AMELIE,4,3/24/11 8:44,4.95,14129.0,UNITED KINGDOM
3,547561,23077,DOUGHNUT LIP GLOSS,20,3/24/11 8:44,1.25,14129.0,UNITED KINGDOM
4,547561,84375,SET OF 20 KIDS COOKIE CUTTERS,12,3/24/11 8:44,2.10,14129.0,UNITED KINGDOM


Let's check if the numerical data is cleaned
* It is cleaned!

In [8]:
df_transformed.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

We now check the mean values from the original data.

In [9]:
df[['Quantity','UnitPrice']].mean()

Quantity     10.985865
UnitPrice     4.258975
dtype: float64

And the learned mean values from `MyCustomTransformerForMeanImputation()` dictionary. We assess the 'custom_transf' steps and check the attribute `.imputer_dict_`, which happens to be the dictionary we stored the mean values in the `.fit()` method.

In [10]:
pipeline['custom_transf'].imputer_dict_

{'Quantity': np.float64(10.985865393616026),
 'UnitPrice': np.float64(4.2589750070205)}